In [1]:
import pandas as pd

# Load the dataset from data folder
df = pd.read_csv('../data/Life Expectancy Data.csv')

# Display the first 5 rows to make sure it loaded correctly
print("--- Dataset Preview ---")
print(df.head())

# Check which columns have missing values (NaNs)
print("\n--- Missing Values Count Per Column ---")
print(df.isnull().sum())

--- Dataset Preview ---
       Country  Year      Status  Life expectancy   Adult Mortality  \
0  Afghanistan  2015  Developing              65.0            263.0   
1  Afghanistan  2014  Developing              59.9            271.0   
2  Afghanistan  2013  Developing              59.9            268.0   
3  Afghanistan  2012  Developing              59.5            272.0   
4  Afghanistan  2011  Developing              59.2            275.0   

   infant deaths  Alcohol  percentage expenditure  Hepatitis B  Measles   ...  \
0             62     0.01               71.279624         65.0      1154  ...   
1             64     0.01               73.523582         62.0       492  ...   
2             66     0.01               73.219243         64.0       430  ...   
3             69     0.01               78.184215         67.0      2787  ...   
4             71     0.01                7.097109         68.0      3013  ...   

   Polio  Total expenditure  Diphtheria    HIV/AIDS         GD

In [3]:
# ==========================
# Data analysis and cleaning
# ==========================

# Strip any hidden whitespaces from column names
df.columns = df.columns.str.strip()

# Drop rows where the target column 'Life expectancy' is missing
df = df.dropna(subset=['Life expectancy'])

# Fill missing numeric values using the mean of each specific country
numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns

# Group by country and fill missing values with the country's average
df[numeric_cols] = df.groupby('Country')[numeric_cols].transform(lambda x: x.fillna(x.mean()))

# Fill any remaining gaps using the median of their respective Status category
df[numeric_cols] = df.groupby('Status')[numeric_cols].transform(lambda x: x.fillna(x.median()))

print("--- Final Clean Check ---")
print(df.isnull().sum())

--- Final Clean Check ---
Country                            0
Year                               0
Status                             0
Life expectancy                    0
Adult Mortality                    0
infant deaths                      0
Alcohol                            0
percentage expenditure             0
Hepatitis B                        0
Measles                            0
BMI                                0
under-five deaths                  0
Polio                              0
Total expenditure                  0
Diphtheria                         0
HIV/AIDS                           0
GDP                                0
Population                         0
thinness  1-19 years               0
thinness 5-9 years                 0
Income composition of resources    0
Schooling                          0
dtype: int64


In [4]:
# Calculate how every numeric column correlates with Life expectancy
correlations = df.corr(numeric_only=True)['Life expectancy'].sort_values(ascending=False)

print("--- Correlation with Life Expectancy ---")
print(correlations)

--- Correlation with Life Expectancy ---
Life expectancy                    1.000000
Schooling                          0.731799
Income composition of resources    0.703099
BMI                                0.564409
Diphtheria                         0.484676
Polio                              0.471485
GDP                                0.448568
Alcohol                            0.409079
percentage expenditure             0.381864
Hepatitis B                        0.332662
Total expenditure                  0.229348
Year                               0.170033
Population                        -0.028981
Measles                           -0.157586
infant deaths                     -0.196557
under-five deaths                 -0.222529
thinness 5-9 years                -0.466401
thinness  1-19 years              -0.472177
HIV/AIDS                          -0.556556
Adult Mortality                   -0.696359
Name: Life expectancy, dtype: float64


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error
from sklearn.preprocessing import StandardScaler # Import the scaler
import pickle

# 1. Define features and target
features = ['Schooling', 'Income composition of resources', 'HIV/AIDS', 'GDP']
X = df[features]
y = df['Life expectancy']

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Train the model on SCALED data
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# 5. Evaluate performance
y_pred = model.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)

print("--- Updated Model Performance ---")
print(f"R-squared (R²) Score: {r2:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} years")

# 6. Save BOTH the model and the scaler so FastAPI can use them together
with open('../model.pkl', 'wb') as model_file:
    pickle.dump(model, model_file)
with open('../scaler.pkl', 'wb') as scaler_file:
    pickle.dump(scaler, scaler_file)

print("\nSuccess: Scaled model saved as 'model.pkl' and scaler saved as 'scaler.pkl'!")

--- Updated Model Performance ---
R-squared (R²) Score: 0.7464
Root Mean Squared Error (RMSE): 4.68 years

Success: Scaled model saved as 'model.pkl' and scaler saved as 'scaler.pkl'!
